# 01_数据清洗

## 目标
- 完成多表字段标准化、缺失值检查、主键关系校验，形成后续分析可直接使用的主分析表。
- 读取olist所有数据表，检查数据质量，处理缺失值和异常值，合并为分析用主表。

## 输入
- olist_orders_dataset.csv、olist_customers_dataset.csv、olist_order_items_dataset.csv、olist_order_payments_dataset.csv、olist_order_reviews_dataset.csv、olist_products_dataset.csv、olist_sellers_dataset.csv
- data/目录下的9个csv文件

## 输出
- 清洗后的主分析表与数据质量检查结果。
- 清洗后的主表 df_master，供后续notebook使用


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

print("环境OK，pandas版本：", pd.__version__)

环境OK，pandas版本： 3.0.1


In [3]:
data_path = "../data/"

orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")
category = pd.read_csv(data_path + "product_category_name_translation.csv")

for name, df in [("orders",orders),("customers",customers),("items",items),
                 ("payments",payments),("reviews",reviews),("products",products),
                 ("sellers",sellers),("category",category)]:
    print(f"{name}: {df.shape[0]}行 x {df.shape[1]}列")

orders: 99441行 x 8列
customers: 99441行 x 5列
items: 112650行 x 7列
payments: 103886行 x 5列
reviews: 99224行 x 7列
products: 32951行 x 9列
sellers: 3095行 x 4列
category: 71行 x 2列


## 步骤1：字段与数据类型检查


In [4]:
print("=== orders表字段 ===")
print(orders.dtypes)
print("\n=== 各表缺失值 ===")
for name, df in [("orders",orders),("customers",customers),
                 ("items",items),("payments",payments),
                 ("reviews",reviews),("products",products)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n【{name}】\n{missing}")
    else:
        print(f"\n【{name}】无缺失值")

=== orders表字段 ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

=== 各表缺失值 ===

【orders】
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

【customers】无缺失值

【items】无缺失值

【payments】无缺失值

【reviews】
review_comment_title      87656
review_comment_message    58247
dtype: int64

【products】
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


## 步骤2：缺失值与重复值处理


In [5]:
orders = orders.dropna(subset=['order_approved_at'])
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print("重复值检查：")
for name, df in [("orders",orders),("items",items),("payments",payments)]:
    print(f"{name} 重复行：{df.duplicated().sum()}")
print("\n缺失值处理完成")

重复值检查：
orders 重复行：0
items 重复行：0
payments 重复行：0

缺失值处理完成


## 步骤3：主键关联与口径统一


In [8]:
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(items, on='order_id', how='left')

pay_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value','sum'),
    payment_installments=('payment_installments','max')
).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')

products = products.merge(category, on='product_category_name', how='left')
df = df.merge(products[['product_id','product_category_name',
                          'product_category_name_english']],
              on='product_id', how='left')

review_agg = reviews.groupby('order_id').agg(
    review_score=('review_score','mean')
).reset_index()
df = df.merge(review_agg, on='order_id', how='left')

# 时间字段转换
for col in ['order_purchase_timestamp','order_approved_at',
            'order_delivered_customer_date','order_estimated_delivery_date']:
    df[col] = pd.to_datetime(df[col])

df['purchase_year'] = df['order_purchase_timestamp'].dt.year
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
df['delivery_days'] = (df['order_delivered_customer_date'] -
                        df['order_purchase_timestamp']).dt.days

print("主表合并完成，shape：", df.shape)
df.head(3)

主表合并完成，shape： (113264, 27)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,freight_value,total_payment,payment_installments,product_category_name,product_category_name_english,review_score,purchase_year,purchase_month,purchase_hour,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,8.72,38.71,1.0,utilidades_domesticas,housewares,4.0,2017,10,10,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,22.76,141.46,1.0,perfumaria,perfumery,4.0,2018,7,20,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,19.22,179.12,3.0,automotivo,auto,5.0,2018,8,8,9.0


## 步骤4：导出清洗后数据


In [9]:
df.to_csv("../data/df_master.csv", index=False, encoding='utf-8-sig')
print("已保存到 data/df_master.csv")
print("shape：", df.shape)
print("\n字段列表：")
print(df.columns.tolist())

已保存到 data/df_master.csv
shape： (113264, 27)

字段列表：
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'total_payment', 'payment_installments', 'product_category_name', 'product_category_name_english', 'review_score', 'purchase_year', 'purchase_month', 'purchase_hour', 'delivery_days']
